# 03 · Pre-registered analysis

The decision rules in this notebook are copied from `preregistration.md`. Do not change `ALPHA`, `SESOI` or the model specifications after seeing the results.

The primary model is a linear probability model with story fixed effects and standard errors clustered by writer. A logit with one dummy per story is avoided, because with only two rows per story it suffers from incidental-parameter bias; conditional logit and McNemar's test are reported as checks.

In [ ]:
import json, re
from pathlib import Path
import numpy as np, pandas as pd
import statsmodels.api as sm
from scipy import stats
from statsmodels.discrete.conditional_models import ConditionalLogit

In [ ]:
RUN_NAME = "gpt-4o-mini"     # the results folder written by notebook 02 ("mock" for the pipeline test)
OUT = Path("results") / RUN_NAME
RAW = OUT / "raw.jsonl"
STORIES = "data/stories.csv"
CUE_SAMPLE = Path("data/validation_cue_sample.csv")

## Decision rules and helper functions

In [ ]:
EMOTIONS = ["joy", "fear", "anger", "sadness", "disgust", "shame", "guilt"]
ALPHA = 0.05      # one-sided
SESOI = 0.03      # smallest effect of interest: 3 percentage points


# ---------- 1. Parsing (an automated measurement: validated by hand below) ----------
def parse(raw):
    if not isinstance(raw, str) or raw == "__ERROR__":
        return "invalid"
    words = re.findall(r"[a-z]+", raw.lower())
    found = [w for w in words if w in EMOTIONS]
    if len(set(found)) == 1:
        return found[0]
    return "invalid"   # refusal, several labels, or a label outside the list: scored incorrect


# ---------- 2. Within-story linear probability model, errors clustered by writer ----------
def within_story_lpm(d, regressors):
    """Story fixed effects by demeaning inside each story; SEs clustered by writer.
    Coefficients are differences in accuracy (percentage points / 100)."""
    X = d[regressors].astype(float)
    y = d["correct"].astype(float)
    Xd = X - X.groupby(d.story_id).transform("mean")
    yd = y - y.groupby(d.story_id).transform("mean")
    fit = sm.OLS(yd, Xd).fit(cov_type="cluster", cov_kwds={"groups": d.writer_id})
    return fit


def one_sided_p(fit, name):
    t = fit.tvalues[name]
    return float(1 - stats.norm.cdf(t))


def decide(effect, p):
    return "YES" if (p < ALPHA and effect >= SESOI) else "NO"


def h1(d):
    """Matched vs mismatched nationality, net of story and persona-country effects."""
    sub = d[d.condition.isin(["matched", "mismatched"])].copy()
    sub["matched"] = (sub.condition == "matched").astype(int)
    pc = pd.get_dummies(sub.persona_country, prefix="pc", drop_first=True, dtype=int)
    sub = pd.concat([sub, pc], axis=1)
    fit = within_story_lpm(sub, ["matched"] + list(pc.columns))
    eff, p = float(fit.params["matched"]), one_sided_p(fit, "matched")
    lo, hi = fit.conf_int(alpha=0.10).loc["matched"]      # 90% CI = one-sided 95% bound
    # McNemar on story pairs, as a transparent check
    wide = sub.pivot_table(index="story_id", columns="condition", values="correct")
    b = int(((wide.matched == 1) & (wide.mismatched == 0)).sum())
    c = int(((wide.matched == 0) & (wide.mismatched == 1)).sum())
    mcn = stats.binomtest(b, b + c, 0.5, alternative="greater").pvalue if b + c else 1.0
    return {"effect_pp": 100 * eff, "p_one_sided": p, "ci90_pp": [100 * lo, 100 * hi],
            "equivalence_rules_out_3pp": bool(hi < SESOI),
            "mcnemar_b_matched_only": b, "mcnemar_c_mismatched_only": c,
            "mcnemar_p_one_sided": float(mcn), "n_stories": int(sub.story_id.nunique()),
            "decision": decide(eff, p)}


def h2(d):
    """Full persona vs matched nationality, and vs the length-matched control."""
    sub = d[d.condition.isin(["matched", "full", "length_matched"]) & d.full_persona_complete].copy()
    sub["full"] = (sub.condition == "full").astype(int)
    sub["length"] = (sub.condition == "length_matched").astype(int)
    fit = within_story_lpm(sub, ["full", "length"])
    eff, p = float(fit.params["full"]), one_sided_p(fit, "full")
    diff = fit.t_test("full - length = 0")
    d_eff = float(np.squeeze(diff.effect))
    d_p = float(1 - stats.norm.cdf(float(np.squeeze(diff.tvalue))))
    ok = p < ALPHA and eff >= SESOI and d_p < ALPHA
    return {"full_vs_matched_pp": 100 * eff, "p_one_sided": p,
            "full_vs_length_pp": 100 * d_eff, "p_full_vs_length": d_p,
            "n_stories": int(sub.story_id.nunique()), "decision": "YES" if ok else "NO"}


In [ ]:
def accuracy_table(d):
    rows = []
    for cond, g in d.groupby("condition"):
        X = np.ones((len(g), 1))
        fit = sm.OLS(g.correct.astype(float), X).fit(cov_type="cluster",
                                                     cov_kwds={"groups": g.writer_id})
        rows.append({"condition": cond, "accuracy": 100 * fit.params.iloc[0],
                     "se": 100 * fit.bse.iloc[0], "invalid_rate": 100 * (g.pred == "invalid").mean(),
                     "n": len(g)})
    return pd.DataFrame(rows).set_index("condition").round(2)


def stereotype_diagnostic(d):
    """Stereotype signature: does a persona shift the label distribution regardless of the story?
    Gold labels are balanced within each country, so a pure shift cannot raise accuracy."""
    def dist(g):
        return g.pred[g.pred != "invalid"].value_counts(normalize=True).reindex(EMOTIONS, fill_value=0)
    base = dist(d[d.condition == "none"])
    out = {}
    for (cond, pc), g in d[d.condition != "none"].groupby(["condition", "persona_country"]):
        out[f"{cond} | persona={pc}"] = round(0.5 * float((dist(g) - base).abs().sum()), 4)
    return {"total_variation_vs_no_persona": out,
            "label_distribution_no_persona": base.round(3).to_dict()}


def ambiguity_split(d):
    """Information pathway predicts gains where the no-persona model is wrong (ambiguous stories)."""
    none = d[d.condition == "none"].set_index("story_id").correct.rename("none_correct")
    sub = d[d.condition.isin(["matched", "mismatched"])].join(none, on="story_id")
    res = {}
    for k, g in sub.groupby("none_correct"):
        acc = g.groupby("condition").correct.mean()
        res["no-persona correct" if k else "no-persona wrong"] = round(
            float(100 * (acc.get("matched", np.nan) - acc.get("mismatched", np.nan))), 2)
    return res


## Load and score

In [ ]:
d = pd.read_json(RAW, lines=True)
st = pd.read_csv(STORIES)[["story_id", "full_persona_complete", "cue_flag"]]
d = d.merge(st, on="story_id", how="left")
d["pred"] = d.raw_output.map(parse)
d["correct"] = (d.pred == d.gold).astype(int)
print(d.model.iloc[0], "|", len(d), "calls |", d.story_id.nunique(), "stories")
if d.model.iloc[0] == "mock":
    print("WARNING: these are random mock answers. They test the pipeline and must never be reported.")

## Accuracy by condition

Standard errors are clustered by writer. The invalid rate counts refusals and answers outside the seven labels; they are scored as incorrect.

In [ ]:
acc = accuracy_table(d)
acc

## H1 · Does the matched nationality help?

In [ ]:
res_h1 = h1(d)
print(f"Effect {res_h1['effect_pp']:.2f} pp, one-sided p = {res_h1['p_one_sided']:.4f}, "
      f"90% CI [{res_h1['ci90_pp'][0]:.2f}, {res_h1['ci90_pp'][1]:.2f}]")
print("Decision:", res_h1["decision"], "| 3 pp effect ruled out:", res_h1["equivalence_rules_out_3pp"])
res_h1

### Robustness: conditional logit

Only stories whose correctness differs between the matched and mismatched conditions carry information here, exactly as in McNemar's test.

In [ ]:
sub = d[d.condition.isin(["matched", "mismatched"])].copy()
sub["matched"] = (sub.condition == "matched").astype(int)
sub["pc"] = (sub.persona_country == sorted(sub.persona_country.unique())[1]).astype(int)
clogit = ConditionalLogit(sub.correct, sub[["matched", "pc"]], groups=sub.story_id).fit(disp=False)
print(clogit.summary())

### Sensitivity: without stories that contain nationality cues

In [ ]:
h1(d[~d.cue_flag.astype(bool)])

## H2 · Does the full persona add something?

In [ ]:
res_h2 = h2(d)
res_h2

## Mechanism diagnostics

The stereotype signature is how far each persona moves the label distribution away from the no-persona distribution. Because gold labels are balanced within each country, a shift on its own cannot raise accuracy. The information signature is the matching gain split by whether the no-persona model got the story right.

In [ ]:
diag = stereotype_diagnostic(d)
display(pd.Series(diag["total_variation_vs_no_persona"], name="TVD vs no persona"))
print("Matching gain by ambiguity (pp):", ambiguity_split(d))

## Validation of the automated measurements

The first run exports `results/validation_parser_sample.csv`. Fill its `hand_label` column with the label you read in `raw_output` (or `invalid`), and fill `hand_cue` in `data/validation_cue_sample.csv`. Then re-run this cell. Compare parser agreement between matched and mismatched rows, and dictionary recall between countries: differences are the kind of error that could create an effect.

In [ ]:
PARSER_SAMPLE = 50    # outputs per condition to hand-code


def parser_sample_path(d, out):
    # the row count is in the name, so a sample exported after the 20-story pilot is never reused for the full run
    return out / f"validation_parser_sample_n{len(d)}.csv"


def validation(d, out, raw_path):
    """Edgar's check: do automated measurement errors correlate with the predictor?"""
    res = {}
    vfile = parser_sample_path(d, out)
    if not vfile.exists():  # export a blind, stratified sample for hand-coding
        samp = pd.concat([g.sample(min(PARSER_SAMPLE, len(g)), random_state=1)
                          for _, g in d.groupby("condition")])
        samp[["key", "condition", "raw_output"]].assign(hand_label="").to_csv(vfile, index=False)
        res["parser"] = f"exported {len(samp)} outputs to hand-code in {vfile.name}"
    else:
        v = pd.read_csv(vfile).dropna(subset=["hand_label"])
        if len(v):
            v["auto"] = v.raw_output.map(parse)
            v["agree"] = v.auto == v.hand_label.str.strip().str.lower()
            res["parser_agreement_by_condition"] = v.groupby("condition").agree.mean().round(3).to_dict()
    cfile = CUE_SAMPLE
    if cfile.exists():
        c = pd.read_csv(cfile).dropna(subset=["hand_cue"])
        if len(c):
            c = c.merge(d[["story_id", "cue_flag"]].drop_duplicates(), on="story_id")
            c["hand_cue"] = c.hand_cue.astype(int).astype(bool)
            by = {}
            for country, g in c.groupby("country"):
                tp = (g.cue_flag & g.hand_cue).sum()
                by[country] = {"recall": round(tp / max(g.hand_cue.sum(), 1), 3),
                               "precision": round(tp / max(g.cue_flag.sum(), 1), 3), "n": len(g)}
            res["cue_dictionary_by_country"] = by
    return res


def ppi_h1(d, out, B=500, seed=0):
    """Prediction-powered correction of the H1 effect (Angelopoulos et al., 2023).

    The parser's correct/incorrect call is the 'prediction' for every row; the hand-coded rows give the
    rectifier mean(hand_correct - parser_correct) within each condition. The corrected accuracy of a
    condition is mean(parser_correct over all rows) + rectifier. The CI comes from a bootstrap over writers
    that resamples both the full data and the hand-coded rows, so it stays valid if parser errors differ
    between matched and mismatched rows.
    """
    vfile = parser_sample_path(d, out)
    if not vfile.exists():
        return None
    v = pd.read_csv(vfile).dropna(subset=["hand_label"])
    v = v[v.condition.isin(["matched", "mismatched"])]
    if v.groupby("condition").size().reindex(["matched", "mismatched"]).fillna(0).min() < 20:
        return None
    v = v.drop(columns="condition").merge(d[["key", "condition", "gold", "writer_id", "correct"]], on="key")
    v["hand_correct"] = (v.hand_label.str.strip().str.lower() == v.gold).astype(int)
    sub = d[d.condition.isin(["matched", "mismatched"])]

    def effect(dd, vv, rectify):
        acc = {}
        for c in ("matched", "mismatched"):
            a = dd[dd.condition == c].correct.mean()
            if rectify:
                r = vv[vv.condition == c]
                a += (r.hand_correct - r.correct).mean()
            acc[c] = a
        return 100 * (acc["matched"] - acc["mismatched"])

    gd = {w: g for w, g in sub.groupby("writer_id")}
    gv = {w: g for w, g in v.groupby("writer_id")}
    writers = np.array(list(gd))
    rng = np.random.default_rng(seed)
    boot = {"naive": [], "ppi": []}
    for _ in range(B):
        w = rng.choice(writers, len(writers))
        dd = pd.concat([gd[x] for x in w])
        vv = pd.concat([gv[x] for x in w if x in gv])
        boot["naive"].append(effect(dd, vv, False))
        boot["ppi"].append(effect(dd, vv, True))
    out_ = {"n_hand_coded": v.groupby("condition").size().to_dict(),
            "parser_error_rate_pp": {c: round(100 * float((g.hand_correct != g.correct).mean()), 2)
                                     for c, g in v.groupby("condition")}}
    for name, rectify in (("naive", False), ("ppi", True)):
        lo, hi = np.percentile(boot[name], [5, 95])
        out_[name] = {"effect_pp": round(effect(sub, v, rectify), 2), "ci90_pp": [round(lo, 2), round(hi, 2)],
                      "ci_excludes_zero": bool(lo > 0)}
    return out_


In [ ]:
val = validation(d, OUT, RAW)
ppi = ppi_h1(d, OUT)
display(val)
print("\nH1 corrected by prediction-powered inference:", ppi or "needs hand-coded matched and mismatched rows (20+ each)")


## Save everything for the slides

In [ ]:
summary = {"model": d.model.iloc[0], "n_calls": len(d),
           "accuracy_by_condition": acc.to_dict(orient="index"),
           "H1_nationality_matching": res_h1,
           "H1_sensitivity_without_cue_stories": h1(d[~d.cue_flag.astype(bool)]),
           "H1_ppi_corrected": ppi,
           "H2_full_persona": res_h2, "stereotype_diagnostic": diag,
           "matching_gain_by_ambiguity_pp": ambiguity_split(d), "validation": val}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2, default=float))
print("Saved", OUT / "summary.json")